# Algorithm 3 — SMC weighted flow maps on the FLUX.1-dev flow map

Sampling $q_\lambda(x) \propto p(x)\exp(\lambda f(x))$ with $f$ = **normalized squared global IEM**, in a
**single forward pass** over a two-timestep flow map — no tempering ladder, no rejuvenation.

**Why.** `adaptive_tempering_smc.py` + `PCNKernel` does not finish at FLUX scale: jobs 695271/697271 burned
~14 h and reached $\beta = 0.5422$ ($m_{\text{eff}}$ 1.63 of 3), with pCN acceptance collapsing
0.75 → 0.62 → 0.38 → 0.12. Every tempering level costs a full ladder of rejuvenation sweeps, and the ladder
never ends. Algorithm 3 replaces it with $N$ steps from noise to data. The question this notebook answers:
**does one ~3 h forward pass reach a tilt strength the pCN ladder could not reach in 14 h?**

**The model.** `gabeguofanclub/flux-1-dev-flowmap-lsd` — a rank-64 LoRA over the FLUX.1-dev transformer plus
a second timestep embedder, giving a velocity $u_\theta(x; t \to t_{to})$ conditioned on a *pair* of times.
Two operations, and they are not interchangeable:

$$\text{map}(x, t, 1) = x - \sigma_{t}u_\theta(x; t \to 1) \quad\text{(ODE endpoint, a sample)} \qquad
  \text{denoise}(x, t) = x - \sigma_{t}u_\theta(x; t \to t) \quad\text{(Tweedie mean, the score model)}$$

`denoise` is **not** `map(x, t, t)` — $X_{t,t}$ is the identity, which multiplies the velocity away. Getting
this wrong fails silently, giving the score of $N(0, \sigma_{t}^{2})$.

**This notebook owns everything that defines $p$** — the checkpoint, the prompt (`"A dog"`), the guidance
scale, the encoded embeddings — and hands only closures to the library, exactly as
`flux_dev_strong_tilt_sweep_2_1.ipynb` does. The library never sees a prompt.

**Phases** (`STOP_AFTER` selects how far to go):

| phase | what | cost |
|---|---|---|
| 1 `health` | model wiring + gate: load validation, latent std, `denoise` $\neq$ `map` | ~20 min |
| 3 `refs` | $R{=}64$ references from an **untilted** run, persisted; frozen reward; $\lambda_s$ on a **held-out** batch | ~45 min |
| 4 `run` | the tilted run, $\lambda = m\lambda_s$, per-step checkpointing | ~3 h per $m$ |
| 5 | decoded grid, diagnostics, Takeaways | ~10 min |

Phase 2 (the sampler and its 43 tests) is CPU-only and lives in `creativity_measure/flowmap_smc.py` /
`tests/test_flowmap_smc.py`; it must be green before any of this runs.

In [1]:
import json
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

T_START = time.time()

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(REPO, "creativity_measure")):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)
RUN_DIR = os.path.join(REPO, "notebooks", "flowmap_smc_flux")
os.makedirs(RUN_DIR, exist_ok=True)

# --- what to run ----------------------------------------------------------------------------------
# The phases are cumulative and each caches to disk, so a later job loads what an earlier one built.
# STOP_AFTER lets one notebook + one .slurm serve the whole execution order: submit with "health"
# first (the gate), then "refs", then "run".
STOP_AFTER = os.environ.get("STOP_AFTER", "run")          # health | refs | run | strength
assert STOP_AFTER in ("health", "refs", "run", "strength"), STOP_AFTER
DO_REFS = STOP_AFTER in ("refs", "run", "strength")
DO_RUN = STOP_AFTER == "run"
# "strength" is PHASE 6: a lambda = 0 A/B of the base transition (see that cell). It needs the reward
# (the mandatory terminal update fires even at lambda = 0), hence DO_REFS, but never the tilted run.
DO_STRENGTH = STOP_AFTER == "strength"
RUN_TAG = os.environ.get("RUN_TAG", "a3_1")

# --- wall clock -----------------------------------------------------------------------------------
# The SMC is one long library call, so it cannot be interrupted from outside; the `on_step` callback
# checks DEADLINE after every step instead. RESERVE_H is held back for decoding and rendering.
# Under Slurm, SLURM_JOB_END_TIME is the job's real end, so the deadline tracks --time automatically.
WALL_BUDGET_H = float(os.environ.get("WALL_BUDGET_H", 6.0))   # keep in sync with --time in the .slurm
RESERVE_H = 0.75                                              # VAE decoding, rendering, writing out
try:
    _end = float(os.environ["SLURM_JOB_END_TIME"])
except (KeyError, ValueError):
    _end = None
if _end and _end > T_START:
    DEADLINE, WALL_BUDGET_H, _src = _end - RESERVE_H * 3600, (_end - T_START) / 3600, "SLURM"
else:
    DEADLINE, _src = T_START + (WALL_BUDGET_H - RESERVE_H) * 3600, "constant"

from creativity_measure import set_default_dtype

set_default_dtype(torch.float32)          # FLUX runs bf16; everything the SMC touches is fp32
torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16                    # transformer compute dtype; particles stay fp32
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"

# --- progress log ----------------------------------------------------------------------------------
# `nbconvert --execute` captures cell stdout into the OUTPUT notebook, which is only written when the
# job ends -- so the Slurm .out log shows nothing for the whole run and there is no way to watch a 9 h
# job from outside. `note()` mirrors the milestones that matter to a plain text file that can be
# tailed live. Line-buffered and flushed per call, so a killed job still leaves everything up to the
# last step on disk.
PROGRESS_PATH = os.path.join(RUN_DIR, f"progress_{RUN_TAG}_{STOP_AFTER}.log")


def note(msg: str) -> None:
    """Print AND append to PROGRESS_PATH, with a wall-clock offset."""
    line = f"[{(time.time() - T_START) / 60:7.1f} min] {msg}"
    print(line, flush=True)
    with open(PROGRESS_PATH, "a") as fh:
        fh.write(line + "\n")
        fh.flush()


from huggingface_hub import auth_check

auth_check("black-forest-labs/FLUX.1-dev")
note(f"START phase={STOP_AFTER} tag={RUN_TAG} job={os.environ.get('SLURM_JOB_ID', 'manual')} "
     f"gpu={torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'}")
print(f"device {DEVICE} ({DTYPE})   budget {WALL_BUDGET_H:.2f} h via {_src}")
print(f"deadline {time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")
print(f"STOP_AFTER={STOP_AFTER!r}  run tag {RUN_TAG!r}   FLUX.1-dev access: OK")

OfflineModeIsEnabled: Cannot reach https://huggingface.co/api/models/black-forest-labs/FLUX.1-dev/auth-check: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.

In [ ]:
from diffusers import FluxPipeline

MODEL_ID = "black-forest-labs/FLUX.1-dev"
# LOAD PATH vs IDENTITY. MODEL_ID stays the canonical repo id because it is recorded in BASE_PROCESS
# and asserted against the persisted reference file -- swapping a local path in there would fail that
# check for no reason. MODEL_PATH is only where the weights are read from: the slurm script stages the
# 32 G snapshot to node-local disk and exports FLUX_LOCAL_PATH, because the NFS mount serves it at
# ~2 MB/s (a MEASURED 257 min cold, job 755539). `from_pretrained` on a directory holding
# model_index.json never resolves a repo id, which is also what makes HF_HUB_OFFLINE=1 safe.
MODEL_PATH = os.environ.get("FLUX_LOCAL_PATH") or MODEL_ID
if MODEL_PATH != MODEL_ID:
    assert os.path.isfile(os.path.join(MODEL_PATH, "model_index.json")), (
        f"FLUX_LOCAL_PATH={MODEL_PATH} has no model_index.json -- the slurm staging step did not "
        f"finish; loading would silently fall back to the hub")
    print(f"loading FLUX from node-local stage: {MODEL_PATH}")
else:
    print("FLUX_LOCAL_PATH unset -- falling back to the HF cache over NFS (~257 min cold)")
PROMPT = "A dog"

# GUIDANCE = 1.0 for the flow map (the pCN sweeps used 1.5 / 3.5 on base FLUX). Together with PROMPT
# and the checkpoint this IS p, so it is fixed here and never exposed to the library.
GUIDANCE = 1.0

IMG = 512
C, H, W = 16, IMG // 8, IMG // 8       # VAE downsamples 8x; FLUX latents have 16 channels
LATENT_DIM = C * H * W                 # 65536 -- the geometry the FLUX reward machinery is tuned for

pipe = FluxPipeline.from_pretrained(MODEL_PATH, torch_dtype=DTYPE).to(DEVICE)
with torch.no_grad():
    prompt_embeds, pooled_prompt_embeds, text_ids = pipe.encode_prompt(
        prompt=PROMPT, prompt_2=None, device=DEVICE, max_sequence_length=512
    )
# Free both text encoders (~9.5 GB, T5 dominates): the embeddings above are all we need.
pipe.text_encoder = pipe.text_encoder_2 = pipe.tokenizer = pipe.tokenizer_2 = None
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
transformer, vae = pipe.transformer.eval(), pipe.vae.eval()
VAE_SF, VAE_SHIFT = vae.config.scaling_factor, vae.config.shift_factor
print(f"loaded {MODEL_ID}  prompt={PROMPT!r}  guidance={GUIDANCE}  "
      f"guidance_embeds={transformer.config.guidance_embeds}")
print(f"latents ({C}, {H}, {W}) = d {LATENT_DIM}  vae scaling={VAE_SF} shift={VAE_SHIFT}")


def decode(flat: torch.Tensor, chunk: int = 2) -> torch.Tensor:
    """Flat FLUX latents (B, d) -> images (B, 3, 512, 512) in [0, 1].

    Chunked because the VAE decoder, not the transformer, is the memory spike.
    """
    outs = []
    for i in range(0, flat.shape[0], chunk):
        lat = flat[i : i + chunk].reshape(-1, C, H, W).to(DEVICE, DTYPE) / VAE_SF + VAE_SHIFT
        with torch.no_grad():
            img = vae.decode(lat).sample
        outs.append(pipe.image_processor.postprocess(img.float(), output_type="pt").cpu())
    return torch.cat(outs)

In [ ]:
from creativity_measure import LinearSchedule, ddpm_step, flow_map_step
from creativity_measure.distances.edm_adapter import chunked_denoiser, edm_score_fn
from creativity_measure.generators.flux_flowmap import (
    FluxFlowMap, flow_map_denoiser, load_flow_map_weights,
)

# --- denoiser batch cap ---------------------------------------------------------------------------
# MEASURED, job 695266 on this exact card/model/resolution (probe_denoiser_batch.py):
#     rows      24     48     96    128    192
#     s/row  0.259  0.251  0.255  0.263    OOM
# s/row is FLAT across the range, so a narrow cap costs nothing in throughput (same number of forward
# passes, same cost each) and leaves headroom for a multi-hour run. R=64 with NUM_EPS=3 needs 192 rows
# for the reference-bank build, which is exactly the width that OOMs on an A6000.
# FIXED for the whole run, so f stays a deterministic function of x (CLAUDE.md invariant 1).
MAX_DENOISER_ROWS = 24
SIGMA_MIN, SIGMA_MAX = 2e-3, 80.0

SCHEDULE = LinearSchedule()               # alpha_t = t, sigma_t = 1 - t; FLUX is rectified flow

# --- load the flow-map weights --------------------------------------------------------------------
# ORDER MATTERS: load_flow_map_weights wraps time_text_embed in a DualTimeEmbedder FIRST, or the
# second-embedder keys have nowhere to land. It asserts every tensor is consumed and that the loaded
# model differs from the unmodified one on a fixed input -- without that check a silently no-op load
# means sampling plain FLUX.1-dev while believing it is the flow map, and nothing downstream catches it.
load_flow_map_weights(transformer)

# max_rows caps the transformer's batch the same way MAX_DENOISER_ROWS caps the score path: the
# lookahead calls map() on M*K = 32 rows and reference generation on R = 64, against a model already
# holding ~24 GB. Fixed for the run -- chunking agrees only to float epsilon, so a varying cap would
# make f non-deterministic in x.
flow_map = FluxFlowMap(
    transformer, prompt_embeds, pooled_prompt_embeds, text_ids,
    guidance=GUIDANCE, img_shape=(C, H, W), dtype=DTYPE, max_rows=MAX_DENOISER_ROWS,
)

# ONE wrapper feeding BOTH the reward's distance and the sampler's S_k: the score model that defines
# f's geometry and the one in the score correction must be the same object on the same code path.
# `denoise`, never `map(., t, 1)` -- only a conditional mean backs a marginal score.
denoiser_capped = chunked_denoiser(flow_map_denoiser(flow_map, SCHEDULE), MAX_DENOISER_ROWS)
score_fn = edm_score_fn(denoiser_capped, (C, H, W))
note(f"flow map weights loaded; score ready (denoiser capped at {MAX_DENOISER_ROWS} rows)")


# --- the base process p ----------------------------------------------------------------------------
# ts, schedule, stoch_window and the two TransitionSteps ARE p. guid_window is deliberately NOT in this
# dict: it only decides where the reward runs, so it can be swept freely against one fixed p and one
# reference set. Changing anything below invalidates the references; changing guid_window does not.
N_STEPS = 16
BASE_PROCESS = {
    "schedule": SCHEDULE.name,
    "n_steps": N_STEPS,
    "ts": None,                                # None => linspace(0, 1, N_STEPS + 1)
    "stoch_window": (0.1, 1.0),
    "inside_step": ddpm_step.__name__,
    "outside_step": flow_map_step.__name__,
    "img_shape": [C, H, W],
    "model_id": MODEL_ID,
    "prompt": PROMPT,
    "guidance": GUIDANCE,
    "max_denoiser_rows": MAX_DENOISER_ROWS,
    "max_flow_map_rows": MAX_DENOISER_ROWS,
}
BASE_KW = dict(
    flow_map=flow_map, score_fn=score_fn, schedule=SCHEDULE, n_steps=N_STEPS,
    stoch_window=BASE_PROCESS["stoch_window"], inside_step=ddpm_step, outside_step=flow_map_step,
)
print("base process p:", json.dumps({k: v for k, v in BASE_PROCESS.items()
                                     if k not in ("model_id",)}, default=str))

In [ ]:
from creativity_measure import LpDistance, Reward, flowmap_smc_sample

# --- PHASE 1 GATE: is p itself healthy? ------------------------------------------------------------
# If p looks degraded, stop before building 3 h on top of it.

S_ROW = 0.206            # s per row, MEASURED on an L40S by job 700483's reference-bank build
                         # (5568 rows in 19.1 min) -- the EXACT reward code path the run spends
                         # ~97% of its time in, so it is the right number to budget with. The
                         # 8-row flow-map path measures slower (0.231) because per-call overhead
                         # is amortised over less work; the A6000 figure was 0.251.
S_BASE_FLUX = 1.0150     # latent std of base FLUX.1-dev at this resolution (job 695271)


def _dummy_reward(n: int = 2) -> Reward:
    """A reward that costs nothing, for drawing from p itself.

    `flowmap_smc_sample` needs a `Reward` to pin d/device/dtype, but at lam = 0 its *value* cannot
    affect the trajectory (V = 0 everywhere, and systematic resampling on uniform weights is the
    identity). An L2 reward against dummy references is pure tensor arithmetic -- no network. Paired
    with `guid_window=(1.0, 1.0)` this skips every lookahead, so an untilted draw costs exactly one
    flow-map row per particle per step and NOT the M(2K+2) a guided step would.
    """
    return Reward(distance=LpDistance(p=2.0),
                  x_refs=torch.zeros(n, LATENT_DIM, device=DEVICE, dtype=torch.float32))


def draw_from_p(n: int, seed: int, *, verbose: bool = False) -> torch.Tensor:
    """n untilted samples from p, through the SAME stepping code path the tilted run uses.

    This is what makes the references valid: they are drawn by the base process the run will sample,
    not by some other integrator.
    """
    res = flowmap_smc_sample(_dummy_reward(), 0.0, n, **BASE_KW,
                             guid_window=(1.0, 1.0), mc_samples=1, seed=seed, verbose=verbose)
    return res.X.detach()


t0_health = t0 = time.time()
health = draw_from_p(8, seed=1, verbose=True)
S_health = float(health.std())
print(f"\nuntilted draw: {tuple(health.shape)} in {time.time() - t0:.0f}s   "
      f"latent std = {S_health:.4f}  (base FLUX: {S_BASE_FLUX:.4f})")
print(f"  implied {(time.time() - t0) / (8 * N_STEPS):.3f} s/row vs {S_ROW} measured")

# --- gate 1: p is not degraded --------------------------------------------------------------------
assert 0.6 * S_BASE_FLUX < S_health < 1.6 * S_BASE_FLUX, (
    f"latent std {S_health:.4f} is far from base FLUX's {S_BASE_FLUX:.4f}: p looks degraded, and the "
    f"reward/lambda calibration built on top of it would be meaningless. Check the flow-map load, the "
    f"time-convention flip in FluxFlowMap._velocity, and GUIDANCE."
)

# --- gate 2: denoise(x, t) and map(x, t, 1) are genuinely different -------------------------------
# If they coincide, the two-timestep conditioning is inert and the load is suspect. denoise is a
# conditional mean, so it must be BLURRIER (lower variance) than the ODE endpoint, which is a sample.
t_probe = 0.5
gen = torch.Generator(device=DEVICE).manual_seed(0)
eps = torch.randn(health.shape, generator=gen, device=DEVICE, dtype=health.dtype)
x_t = SCHEDULE.alpha(t_probe) * health + SCHEDULE.sigma(t_probe) * eps
mean_est = flow_map.denoise(x_t, t_probe)
sample_est = flow_map.map(x_t, t_probe, 1.0)
rel_gap = float((mean_est - sample_est).norm() / sample_est.norm())
print(f"\nat t={t_probe}:  denoise std {float(mean_est.std()):.4f}   map(.,t,1) std "
      f"{float(sample_est.std()):.4f}   relative gap {rel_gap:.3f}")
assert rel_gap > 1e-3, (
    "denoise(x, t) == map(x, t, 1): the two-timestep conditioning is inert. Either the DualTimeEmbedder "
    "is not receiving a timestep pair, or the LoRA load was a no-op."
)
assert float(mean_est.std()) < float(sample_est.std()), (
    "the Tweedie mean is not blurrier than the ODE endpoint -- denoise and map are likely swapped"
)
assert torch.allclose(flow_map.map(x_t, t_probe, t_probe), x_t), "X_{t,t} must be the identity"

fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax, img in zip(axes, decode(health)):
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
fig.suptitle(f"PHASE 1 — untilted draws from p   (std {S_health:.4f}, prompt {PROMPT!r})")
plt.savefig(os.path.join(RUN_DIR, f"phase1_health_{RUN_TAG}.png"), dpi=90, bbox_inches="tight")
plt.show()
note(f"PHASE 1 GATE PASSED  latent_std={S_health:.4f} (base FLUX {S_BASE_FLUX:.4f})  "
     f"denoise-vs-map gap={rel_gap:.4f}  s/row={(time.time() - t0_health) / (8 * N_STEPS):.3f}")
if STOP_AFTER == "health":
    print("STOP_AFTER=health -- stopping here.")

### Phase 1 conclusions — gate passed (job 700463, L40S, 27 min)

| check | result | verdict |
|---|---|---|
| LoRA load | 692 tensors, 346 LoRA sites, $\max\lvert\Delta\rvert = 1.41$ on a fixed input | **not a no-op** — this is the flow map, not plain FLUX.1-dev |
| latent std | **1.0345** vs base FLUX's known-good 1.0150 | $p$ is not degraded |
| `denoise` vs `map(·,t,1)` at $t=0.5$ | std 0.9611 vs 0.9961, relative gap 0.166 | genuinely distinct, and the conditional **mean is blurrier than the sample** — exactly the ordering that would break if the two were conflated |
| $X_{t,t}$ | identity to float precision | flow-map algebra is right |
| decoded images | 8 clean, on-prompt dogs | the two-timestep conditioning produces real samples |
| throughput | **0.208 s/row** | 17% *faster* than the A6000's 0.251 the plan budgeted with |

**Three things this run changed downstream.**

1. **`S_ROW` 0.251 → 0.208.** All cost projections and the per-$m$ deadline check now use the measured
   L40S number. At this speed all of $m \in \{1,2,3\}$ fits in a 9 h job, so **`K` stays at the plan's
   operating point of 4** — no need to trade lookahead quality for wall clock.
2. **Setup costs ~22 min of every job**, essentially all of it loading ~34 GB of FLUX weights off the
   shared NFS volume (measured: 4% CPU, 51 s of CPU time in 20 min — I/O-bound, not compute). Budget it.
3. **`FluxFlowMap` now chunks its forward pass** (`max_rows=24`). The gate ran at $B=8$ and never
   exposed this, but reference generation calls `map` on $R = 64$ rows and the lookahead on $M K = 32$,
   against a transformer already holding ~24 GB. Chunking is *mathematically* identical but agrees only
   to float epsilon (measured $2.5\times10^{-7}$ relative), because BLAS picks different kernels per
   batch size — so `max_rows` is fixed for the run, like the denoiser cap, or $f$ would stop being a
   deterministic function of $x$.

**Not verified here, by design:** that $\text{denoise}$ is the *correct* Tweedie mean rather than merely
a different, blurrier function. Nothing cheap distinguishes those. What backs it is the algebra
(`test_flowmap_smc.py::test_score_at_matches_the_closed_form_and_tweedie`, exact to float64 on an
analytic model) plus the $\sigma$-ordering above. The reward's own sanity check in Phase 3 —
$f(\text{refs}) \approx (R-1)/R$ — is the next real constraint on the score field.

In [ ]:
from creativity_measure import SquaredGlobalIEMDistance
from creativity_measure.tilt import NormalizedExpectedDistanceReward

# --- PHASE 3: the frozen reward --------------------------------------------------------------------
# These constants DEFINE f, so changing any of them changes the target. Seeds are load-bearing:
# distance seed 123, reference seed 7 (both carried over from the validated run 695271).
R, N_GAMMA, NUM_EPS = 64, 30, 3
REF_SEED, DIST_SEED, HELDOUT_SEED = 7, 123, 8
N_HELDOUT = 32

REFS_PATH = os.path.join(RUN_DIR, f"refs_R{R}_seed{REF_SEED}.pt")
LAMS_PATH = os.path.join(RUN_DIR, f"lam_s_R{R}_seed{REF_SEED}.json")


def _assert_same_p(saved: dict) -> None:
    """The reference file is keyed to the base process -- but NOT to `guid_window`.

    References must be generated by the same stepping the run uses, so a mismatch means f describes a
    different p than the one being sampled. `guid_window` is deliberately absent: it may vary freely
    across runs sharing one reference set, which is what makes a window sweep a fair comparison.
    """
    keyed = ("schedule", "n_steps", "ts", "stoch_window", "inside_step", "outside_step",
             "img_shape", "model_id", "prompt", "guidance", "max_denoiser_rows",
             "max_flow_map_rows")
    for k in keyed:
        got, want = saved["base_process"].get(k), BASE_PROCESS[k]
        if isinstance(want, tuple):
            got = tuple(got) if got is not None else got
        assert got == want, f"reference file was built under {k}={got!r}, this run uses {want!r}"
    assert "guid_window" not in saved["base_process"], "guid_window must not be keyed into p"


if DO_REFS:
    if os.path.exists(REFS_PATH):
        saved = torch.load(REFS_PATH, map_location="cpu", weights_only=False)
        _assert_same_p(saved)
        latent_refs = saved["x_refs"].to(DEVICE)
        heldout = saved["heldout"].to(DEVICE)
        print(f"references LOADED from {os.path.basename(REFS_PATH)} "
              f"(built on {saved.get('gpu', '?')}) -- the model is out of the reproducibility chain")
    else:
        # WHY THIS IS PERSISTED. Job 697271 aborted because references regenerated on a different GPU
        # shifted f by 16% of std_p(f) -- enough to change weight ratios by 1.25x. Saving the LATENTS
        # takes the model out of the chain, so every future leg gets a bit-identical reference set.
        t0 = time.time()
        latent_refs = draw_from_p(R, seed=REF_SEED)
        heldout = draw_from_p(N_HELDOUT, seed=HELDOUT_SEED)
        torch.save({"x_refs": latent_refs.detach().cpu(), "heldout": heldout.detach().cpu(),
                    "R": R, "seed": REF_SEED, "heldout_seed": HELDOUT_SEED,
                    "latent_dim": LATENT_DIM, "base_process": BASE_PROCESS,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"},
                   REFS_PATH)
        print(f"references GENERATED in {(time.time() - t0) / 60:.1f} min and saved to "
              f"{os.path.basename(REFS_PATH)} -- future legs will load these")

    S = latent_refs.std().item()
    GAMMA_LO = max(1.0 / S**2, 1.0 / SIGMA_MAX**2)
    GAMMA_HI = min(2.0**10, 1.0 / SIGMA_MIN**2)
    GAMMAS = torch.logspace(math.log2(GAMMA_LO), math.log2(GAMMA_HI), N_GAMMA, base=2).to(DEVICE)

    D2 = SquaredGlobalIEMDistance(None, GAMMAS, num_eps=NUM_EPS, seed=DIST_SEED, score_fn=score_fn)
    t0 = time.time()
    reward = NormalizedExpectedDistanceReward(distance=D2, x_refs=latent_refs)   # builds the ref bank
    t_bank = time.time() - t0
    note(f"reward built: refs {tuple(latent_refs.shape)}  S={S:.4f}  gamma=[{GAMMA_LO:.4g}, {GAMMA_HI:.3g}] "
          f"x {N_GAMMA}   ref bank {t_bank / 60:.1f} min")

    # --- lambda_s: measured once, then CACHED --------------------------------------------------------
    # The two checks below cost (N_GAMMA-1)*NUM_EPS*(R + N_HELDOUT) = 8352 rows ~ 35 min. They are a
    # property of the (frozen) reward, so the run job loads the cached scalars instead of paying again.
    # `_assert_same_p` on the cache is what stops a stale lambda_s from silently calibrating a
    # different p. Delete lam_s_*.json to force a re-measure.
    cached = None
    if os.path.exists(LAMS_PATH):
        with open(LAMS_PATH) as fh:
            cached = json.load(fh)
        _assert_same_p(cached)
        assert cached["R"] == R and cached["n_gamma"] == N_GAMMA and cached["num_eps"] == NUM_EPS
        assert cached["dist_seed"] == DIST_SEED

    # A cache built on other hardware is NOT automatically trustworthy (see the "gpu" note where it is
    # written). Verify on a small subset rather than assume: 8 held-out samples = 696 rows, ~3 min.
    if cached is not None and cached.get("gpu") not in (None, GPU_NAME):
        vals = cached.get("f_heldout_values")
        if vals is None:
            print(f"\ncache was built on {cached.get('gpu')!r}, this job is on {GPU_NAME!r}, and it "
                  f"predates per-sample logging -- re-measuring lambda_s from scratch.")
            cached = None
        else:
            n_chk = min(8, len(vals))
            f_chk = reward(heldout[:n_chk])
            want = torch.tensor(vals[:n_chk], device=f_chk.device, dtype=f_chk.dtype)
            drift = float((f_chk - want).abs().max()) / cached["f_heldout_std"]
            note(f"cross-GPU check ({cached.get('gpu')} -> {GPU_NAME}): max|df| = "
                 f"{drift:.3f} x std_p(f) over {n_chk} held-out samples")
            if drift > 0.5:
                raise RuntimeError(
                    f"f drifted {drift:.2f}x std_p(f) across GPUs -- lambda_s = {cached['lam_s']:.2f} "
                    f"describes a different reward than this job computes. Delete "
                    f"{os.path.basename(LAMS_PATH)} to re-measure, or pin --constraint to "
                    f"{cached.get('gpu')!r}.")
            if drift > 0.15:
                print(f"  WARNING: drift is material ({drift:.2f}x std_p(f)); lambda_s is approximate.")

    if cached is not None:
        lam_s = cached["lam_s"]
        F_P_MEAN, F_P_STD = cached["f_heldout_mean"], cached["f_heldout_std"]
        note(f"lambda_s LOADED from {os.path.basename(LAMS_PATH)}: {lam_s:.3f}  "
              f"(E_p[f] = {F_P_MEAN:.4f} +- {F_P_STD:.4f}, f(refs) = {cached['f_refs_mean']:.4f})")
    else:
        # --- reward sanity: f on its OWN references centres near (R-1)/R, not 1 ---------------------
        # The numerator averages over all R refs INCLUDING the self-pair zero; the denominator is the
        # off-diagonal pair mean, which drops it. A value near 1.0 would mean the self-pair is being
        # excluded somewhere, i.e. f is not the function the calibration assumes.
        t0 = time.time()
        f_refs = reward(latent_refs)
        print(f"\nf(refs): mean {float(f_refs.mean()):.4f}  (expected ~(R-1)/R = {(R - 1) / R:.4f})   "
              f"std {float(f_refs.std()):.4f}   [{(time.time() - t0) / 60:.1f} min]")
        assert abs(float(f_refs.mean()) - (R - 1) / R) < 0.10, \
            "f on its own refs is off the expected centre"

        # --- lambda_s on a HELD-OUT batch, never on the references themselves -----------------------
        t0 = time.time()
        f_heldout = reward(heldout)
        F_P_MEAN, F_P_STD = float(f_heldout.mean()), float(f_heldout.std())
        lam_s = 1.0 / F_P_STD
        print(f"held-out f: mean {F_P_MEAN:.4f}  std {F_P_STD:.4f}  "
              f"({N_HELDOUT} samples, {(time.time() - t0) / 60:.1f} min)")
        note(f"lambda_s = 1/std_p(f) = {lam_s:.3f}  (E_p[f] = {F_P_MEAN:.4f} +- {F_P_STD:.4f})")
        with open(LAMS_PATH, "w") as fh:
            json.dump({"lam_s": lam_s, "f_heldout_mean": F_P_MEAN, "f_heldout_std": F_P_STD,
                       "f_refs_mean": float(f_refs.mean()), "S": S,
                       "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS,
                       "dist_seed": DIST_SEED, "base_process": BASE_PROCESS,
                       # Recorded so a later job on DIFFERENT hardware can VERIFY rather than trust.
                       # f is produced by running the bf16 transformer, so a different card changes the
                       # score field even though the reference latents come byte-identical off disk.
                       # Job 697271 died exactly here: a different GPU shifted f by 16% of std_p(f).
                       "gpu": GPU_NAME,
                       "f_heldout_values": [float(v) for v in f_heldout]}, fh, indent=2,
                      default=str)
        print(f"saved -> {os.path.basename(LAMS_PATH)}")
if STOP_AFTER == "refs":
    print("\nSTOP_AFTER=refs -- stopping here.")

### Phase 3 conclusions — reward frozen (job 700483, L40S, 36 min)

| quantity | value | reading |
|---|---|---|
| references | $R=64$ at $d=65536$, seed 7; held-out 32 at seed 8 | drawn by the **same stepping** the run uses, then persisted — the model is now out of the reproducibility chain |
| $S$ (latent std of refs) | 1.0606 | vs 1.0345 on the 8-sample gate batch and 1.0150 for base FLUX; 64 samples is the better estimate |
| $\gamma$ grid | $[0.889,\ 1.02\times10^{3}]\times 30$ | inside the denoiser's validated range |
| $f(\text{refs})$ | **0.984375** | $= 63/64 = (R-1)/R$ **exactly** — see below |
| $E_p[f]$ (held-out) | $1.0062 \pm 0.0071$ | $f$ is normalized, so $\approx 1$ under $p$ is correct by construction |
| $\lambda_s = 1/\operatorname{std}_p(f)$ | **140.093** | so $m=3$ means $\lambda = 420.28$ and $\lambda\cdot\operatorname{std}_p(f) = 3$ |
| ref-bank throughput | 5568 rows in 19.1 min | **0.206 s/row** on the reward path |

**The $f(\text{refs}) = 63/64$ check is stronger than "approximately right" — it is an exact identity.**
$f(x_r)$ divides by the *off-diagonal* pair mean but its own numerator averages over all $R$ references
**including the self-pair** $D^2(x_r,x_r) = 0$, so $\frac{1}{R}\sum_r f(x_r) = \frac{R-1}{R}$ algebraically,
with no approximation. Landing on 0.984375 to every digit therefore confirms three things at once: the
self-pair distances are *exactly* zero (the score field is deterministic — the same point gives the same
score), the off-diagonal masking in `reference_pair_mean` is correct, and the normalization is wired the
way the $\lambda$ calibration assumes. A value near 1.0 instead would have meant the self-pair was being
excluded somewhere and every $\lambda$ below would be miscalibrated.

**$f$ is extremely concentrated under $p$**: a coefficient of variation of 0.7% ($1.0062 \pm 0.0071$).
That is why $\lambda_s$ is large in absolute terms (140, against 92.2 for the pCN run's base-FLUX $p$ —
not directly comparable, since $p$ differs). What matters is the product: at $m=3$, $\lambda f$ spans
about 3 across the cloud, which is the weight spread `logsumexp` was chosen to handle.

**Caveat worth carrying:** $\operatorname{std}_p(f)$ is estimated from **32** held-out samples, so
$\lambda_s$ itself carries roughly $\pm 13\%$ ($1/\sqrt{2(n-1)}$). The $m$-sweep spans a factor of 3, so
this does not change which $m$ gets selected, but do not read $\lambda_s$ as a precise constant.

**Cost measured here fixes Phase 4's one open parameter.** At 0.206 s/row (and 0.231 on the narrower
flow-map path) the two sanctioned guidance windows differ by one guided step and by whether the third
$\lambda$ survives:

| `guid_window` | guided steps | h per $m$ | all three $m$ in 9 h? |
|---|---|---|---|
| $(0.1, 1.0)$ | 14 | 2.34 – 2.62 | **no** — third dropped at the pessimistic rate |
| $(0.1, 0.9)$ | 13 | 2.17 – 2.44 | **yes**, at both rates |

So Phase 4 runs $(0.1, 0.9)$. Because `guid_window` is deliberately **not** part of $p$, this needs no
reference rebuild and a later run can widen it back and still compare fairly against these same
references — which is the entire reason it was kept out of the base process.

In [ ]:
# --- PHASE 4: the tilted run ------------------------------------------------------------------------
M = 8                                        # particles
K = 4                                        # lookahead draws per particle per guided step
ETA = 1.5                                    # SNR factor: g(t') = eta*g(t)
# Run a3_1 (job 700517) showed this is THE binding constraint, ahead of lambda: at 1.0 every one of
# the 13 guided steps resampled, and 13 consecutive systematic resamplings on M=8 particles cannot
# preserve ancestry at ANY lambda -- all three collapsed to one surviving ancestor of eight. From that
# run's m=1 trace, steps 3-16 sat at ESS/M 0.5-1.0, so a 0.5 threshold should fire only ~2-3 times.
ESS_THRESHOLD = 0.5
# WHERE THE REWARD RUNS. FULL window: guidance on EVERY step, including the first (t: 0 -> 0.0625).
# a3_1 used (0.1, 0.9) purely to buy wall clock, which is a confound while the method is still not
# producing a usable sample. The window stays full until results are good; narrowing it (and the speed
# that buys) is a later optimisation.
#
# (0.0, 1.0) is safe even though t = 0 is degenerate (g = inf, c = 0, gamma = 0): guidance is keyed on
# each step's TARGET time, which is the level the lookahead runs at, and on a grid starting at 0 that
# is never 0. The earliest reachable level here is t = 0.0625, where t' = 0.0516, c = 0.057,
# gamma = 0.0030 and the renoise variance is 0.30 -- all well conditioned. The sampler asserts the
# real invariant (no guided step may land on t = 0) rather than banning a zero lower bound.
#
# COST: 16 guided steps (15 lookahead + the terminal) instead of 13. ~2.50 h per lambda on an L40S
# (0.206 s/row) and ~3.05 h on an A6000 (0.251) -- so all three m fit a 9 h job on an L40S, and two
# fit on an A6000. The loop's deadline check drops whatever does not fit, m=3 first.
#
# No reference rebuild: guid_window is deliberately NOT part of p, so this reuses refs_R64_seed7.pt
# and the cached lambda_s, and stays directly comparable to a3_1.
GUID_WINDOW = (0.0, 1.0)
# ORDER MATTERS, and it is most-important-first -- which a3_1 INVERTED. That run ordered 3,2,1 on the
# assumption the strongest tilt was the headline result; it then collapsed at every m. The priority now
# is getting ANY non-degenerate sample, and the mildest tilt is the likeliest to give one, so m=1 runs
# first. The loop stops as soon as an m does not fit the remaining budget, so this is also what gets
# dropped last. The diagnostics cell sorts by m, so the presentation order is unaffected.
M_LIST = [float(x) for x in os.environ.get("M_LIST", "1,2,3").split(",")]
SEED = 101


class DeadlineReached(RuntimeError):
    """Raised from the `on_step` callback when the wall clock runs out, to stop the SMC cleanly."""


def make_checkpointer(path: str, config: dict):
    """Persist the cloud after every step, and stop the run at the deadline.

    The SMC is one long library call, so `on_step` is the only place a notebook can interrupt it.
    The callback is read-only and consumes no randomness, so the run is bit-identical to one without.
    Writes are temp-file + os.replace, so a kill mid-write cannot corrupt the last good file.
    """
    def on_step(i, snap, partial):
        payload = {
            "step": i, "t": snap.t,
            "X": snap.X, "U": snap.U, "V": snap.V, "ancestors": snap.ancestors,
            "ess_history": partial.ess_history, "resampled_history": partial.resampled_history,
            "uniq_history": partial.uniq_history, "t_history": partial.t_history,
            "t_prime_history": partial.t_prime_history, "guided_history": partial.guided_history,
            "f_mean": partial.f_mean, "f_std": partial.f_std,
            "f_min": partial.f_min, "f_max": partial.f_max,
            "l_mean": partial.l_mean, "l_std": partial.l_std,
            "s_mean": partial.s_mean, "s_std": partial.s_std,
            "config": config, "partial": True,
        }
        tmp = f"{path}.tmp"
        torch.save(payload, tmp)
        os.replace(tmp, path)
        elapsed = (time.time() - config["t0"]) / 60
        note(f"  m={config['m_tilt']:g} step {i + 1}/{N_STEPS} t={snap.t:.4f}  "
             f"ESS/M={partial.ess_history[-1] / config['M']:.2f}  "
             f"uniq/M={partial.uniq_history[-1]:.2f}  f={partial.f_mean[-1]:.4f}  "
             f"{elapsed:.1f} min into this m")
        if time.time() > DEADLINE:
            raise DeadlineReached(f"deadline after step {i + 1} (t={snap.t:.4f})")
    return on_step


# --- cost projection ---------------------------------------------------------------------------
# One reward call on B latents costs (N_GAMMA-1)*NUM_EPS*B denoiser rows. At B = M*K = 32 that is the
# whole run: the reward is ~97% of a step, against M(2K+2) flow-map rows for the network itself.
if DO_RUN:
    _ts = [n / N_STEPS for n in range(N_STEPS + 1)]
    _guided = [n for n in range(N_STEPS - 1) if GUID_WINDOW[0] <= _ts[n + 1] <= GUID_WINDOW[1]]
    rew_rows = (N_GAMMA - 1) * NUM_EPS * M * K
    net_rows = M * (2 * K + 2)
    t_guided = (rew_rows + net_rows) * S_ROW
    t_terminal = ((N_GAMMA - 1) * NUM_EPS * M + M) * S_ROW
    t_unguided = M * S_ROW
    t_run = len(_guided) * t_guided + t_terminal + (N_STEPS - 1 - len(_guided)) * t_unguided
    budget_s = DEADLINE - time.time()
    print(f"projected per tilted run: {len(_guided)} guided steps x {t_guided / 60:.1f} min "
          f"+ terminal {t_terminal / 60:.1f} min  =  {t_run / 3600:.2f} h")
    print(f"  reward is {rew_rows / (rew_rows + net_rows):.1%} of a guided step "
          f"({rew_rows} reward rows vs {net_rows} flow-map rows at {S_ROW} s/row)")
    print(f"budget left {budget_s / 3600:.2f} h -> {budget_s / t_run:.1f} of the "
          f"{len(M_LIST)} requested m values ({M_LIST})")

results, run_paths = {}, {}
if DO_RUN:
    for m_tilt in M_LIST:
        lam = m_tilt * lam_s
        tag = f"N{M}_K{K}_m{m_tilt:g}_seed{SEED}_{RUN_TAG}"
        path = os.path.join(RUN_DIR, f"partial_{tag}.pt")
        config = {
            "M": M, "K": K, "eta": ETA, "n_steps": N_STEPS, "lam": lam, "m_tilt": m_tilt,
            "lam_s": lam_s, "seed": SEED, "ess_threshold": ESS_THRESHOLD,
            "guid_window": GUID_WINDOW, "base_process": BASE_PROCESS,
            "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS, "dist_seed": DIST_SEED,
            "refs_file": os.path.basename(REFS_PATH), "t0": time.time(),
        }
        if time.time() + t_run > DEADLINE:
            print(f"\nskipping m={m_tilt:g}: {t_run / 3600:.2f} h projected does not fit in the "
                  f"{(DEADLINE - time.time()) / 3600:.2f} h left")
            break
        note(f"=== m={m_tilt:g}  lambda={lam:.2f}  (lambda_s={lam_s:.2f})  "
             f"M={M} K={K} N={N_STEPS} eta={ETA} guid_window={GUID_WINDOW} ===")
        t0, timed_out = time.time(), False
        try:
            res = flowmap_smc_sample(
                reward, lam, M, **BASE_KW, mc_samples=K, eta=ETA, guid_window=GUID_WINDOW,
                ess_threshold=ESS_THRESHOLD, antithetic=True, keep_steps=True,
                on_step=make_checkpointer(path, config), seed=SEED, verbose=True,
            )
        except DeadlineReached as exc:
            timed_out = True
            note(f"*** STOPPED ON DEADLINE: {exc}")
            res = None
        mins = (time.time() - t0) / 60
        if res is not None:
            final = os.path.join(RUN_DIR, f"result_{tag}.pt")
            torch.save({"X": res.X.cpu(), "logw": res.logw.cpu(),
                        "steps": [{"t": s.t, "X": s.X, "U": s.U, "V": s.V,
                                   "ancestors": s.ancestors} for s in (res.steps or [])],
                        "ess_history": res.ess_history, "resampled_history": res.resampled_history,
                        "uniq_history": res.uniq_history, "t_history": res.t_history,
                        "t_prime_history": res.t_prime_history, "guided_history": res.guided_history,
                        "f_mean": res.f_mean, "f_std": res.f_std, "f_min": res.f_min,
                        "f_max": res.f_max, "l_mean": res.l_mean, "l_std": res.l_std,
                        "s_mean": res.s_mean, "s_std": res.s_std,
                        "config": {**config, "wall_clock_min": mins}, "partial": False}, final)
            results[m_tilt], run_paths[m_tilt] = res, final
            note(f"m={m_tilt:g} COMPLETED in {mins:.1f} min ({mins / 60:.2f} h)  "
                  f"vs projected {t_run / 3600:.2f} h   saved -> {os.path.basename(final)}")
        else:
            print(f"m={m_tilt:g} incomplete after {mins:.1f} min; partial state in "
                  f"{os.path.basename(path)}")
            break

In [ ]:
# --- PHASE 5: diagnostics ----------------------------------------------------------------------------
# Per CLAUDE.md, select the largest lambda whose uniq/N and min ESS/N stay above ~0.5.
# E_q[f] rises forever and is NOT the stopping signal -- degeneracy is.
if results:
    rows = []
    for m_tilt, res in sorted(results.items()):
        f_final = reward(res.X)
        w = torch.softmax(res.logw, dim=0)
        rows.append({
            "m": m_tilt, "lam": m_tilt * lam_s,
            "E_q[f]": float((w * f_final).sum()), "f_min": float(f_final.min()),
            "f_max": float(f_final.max()),
            "min ESS/M": min(res.ess_history) / M, "final ESS/M": res.ess_history[-1] / M,
            "uniq/M (final)": res.uniq_history[-1],
            "min uniq/M": min(res.uniq_history),
        })
    print(f"{'m':>4} {'lambda':>8} {'E_q[f]':>8} {'f_min':>7} {'f_max':>7} "
          f"{'minESS/M':>9} {'ESS/M':>7} {'uniq/M':>7} {'minuniq':>8}   verdict")
    for r in rows:
        ok = r["min ESS/M"] > 0.5 and r["uniq/M (final)"] > 0.5
        print(f"{r['m']:4.0f} {r['lam']:8.2f} {r['E_q[f]']:8.4f} {r['f_min']:7.4f} {r['f_max']:7.4f} "
              f"{r['min ESS/M']:9.2f} {r['final ESS/M']:7.2f} {r['uniq/M (final)']:7.2f} "
              f"{r['min uniq/M']:8.2f}   {'OK' if ok else 'DEGENERATE'}")
    print(f"\nreference point: E_p[f] on the held-out batch = {F_P_MEAN:.4f} +- {F_P_STD:.4f}")
    print(f"pCN ladder (jobs 695271+697271, ~14 h): reached m_eff = 1.63 of 3")

    # --- per-step diagnostics ------------------------------------------------------------------------
    n_m = len(results)
    fig, axes = plt.subplots(3, n_m, figsize=(5.5 * n_m, 11), squeeze=False)
    for j, (m_tilt, res) in enumerate(sorted(results.items())):
        t = res.t_history
        axes[0][j].plot(t, [e / M for e in res.ess_history], "o-", label="ESS/M")
        axes[0][j].plot(t, res.uniq_history, "s-", label="uniq/M")
        axes[0][j].axhline(0.5, color="r", ls=":", label="degeneracy floor")
        axes[0][j].set_title(f"m={m_tilt:g}  lambda={m_tilt * lam_s:.1f}")
        axes[0][j].set_ylim(0, 1.05); axes[0][j].legend(fontsize=8); axes[0][j].set_xlabel("t")

        axes[1][j].errorbar(t, res.f_mean, yerr=res.f_std, fmt="o-", capsize=2, label="f (lookahead)")
        axes[1][j].axhline(F_P_MEAN, color="k", ls="--", label="E_p[f]")
        axes[1][j].set_ylabel("reward"); axes[1][j].legend(fontsize=8); axes[1][j].set_xlabel("t")

        # The twist terms: at K=4 it is a live question whether L-hat / S-hat move the weights at all.
        axes[2][j].plot(t, res.l_mean, "o-", label="L (log-lik)")
        axes[2][j].plot(t, res.s_mean, "s-", label="S (score corr.)")
        axes[2][j].set_yscale("symlog"); axes[2][j].legend(fontsize=8)
        axes[2][j].set_xlabel("t"); axes[2][j].set_ylabel("raw term (mean)")
    fig.suptitle(f"Algorithm 3 diagnostics — M={M} K={K} N={N_STEPS} eta={ETA} "
                 f"guid_window={GUID_WINDOW}")
    fig.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, f"diagnostics_{RUN_TAG}.png"), dpi=110, bbox_inches="tight")
    plt.show()

    # --- decoded grid: one row per m, plus the untilted baseline -------------------------------------
    grid_rows = [("untilted (p)", health)] + [
        (f"m={m_tilt:g}  lam={m_tilt * lam_s:.0f}", res.X) for m_tilt, res in sorted(results.items())
    ]
    fig, axes = plt.subplots(len(grid_rows), M, figsize=(2.2 * M, 2.4 * len(grid_rows)),
                             squeeze=False)
    for i, (label, X) in enumerate(grid_rows):
        imgs = decode(X[:M])
        for j in range(M):
            axes[i][j].imshow(imgs[j].permute(1, 2, 0).numpy()); axes[i][j].axis("off")
        axes[i][0].set_ylabel(label, fontsize=9)
        axes[i][0].axis("on"); axes[i][0].set_xticks([]); axes[i][0].set_yticks([])
    fig.suptitle(f"Algorithm 3 on FLUX.1-dev flow map — {PROMPT!r}, guidance {GUIDANCE}")
    fig.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, f"tilt_grid_{RUN_TAG}.png"), dpi=110, bbox_inches="tight")
    plt.show()
else:
    print("no completed tilted runs to render "
          f"(STOP_AFTER={STOP_AFTER!r}); phase-1 images and any partial checkpoints are on disk.")
note(f"DONE phase={STOP_AFTER}  total wall clock {(time.time() - T_START) / 3600:.2f} h  completed m values: {sorted(results)}")

In [ ]:
# --- PHASE 6: does the base transition regenerate SEMANTIC diversity? --------------------------------
# WHY. a3_1/a3_2/nv_1 all end with `uniq/N` predicting the number of distinct images exactly (a3_2 m=1
# ends at 0.375 = 3 ancestors, and the grid shows 3 scenes then 6 copies). Yet those copies sit 188-191
# apart in latent space, against ~362 for independent draws -- so `ddpm_step` DOES move clones apart,
# just not anywhere that changes the picture.
#
# The mechanism: `ddpm_step` substitutes the ODE endpoint `map(x,t,1)` for a draw from p(x_1|x_t). By
# the law of total variance the step is then short by c_z^2 * Cov(x_1|x_t) -- the denoising posterior
# covariance, i.e. variance along "which image could this still become". What it does inject,
# sigma_cond * eps, is isotropic, hence largely off-manifold and contracted away by the next map call.
# `posterior_z_ddpm_step` synthesises the missing draw by renoising to t_z and mapping back, at ZERO
# extra network calls (see its docstring for the closed form and why one knob fixes both moments).
#
# WHAT THIS PHASE IS NOT. It does not test the tilt. lambda = 0 throughout, and guid_window=(1.0,1.0)
# so only the mandatory terminal update runs -- ONE reward call on M rows per configuration instead of
# the usual 2784 rows per step. Minutes, not hours.
#
# THE RISK IT EXISTS TO MEASURE. strength=1.0 is exact for GAUSSIAN data. A real posterior in
# d = 65536 is far tighter, so 1.0 is an upper bound and may over-renoise -- which would show up as
# latent std drifting off FLUX's known-good 1.0150 and as E_p[f] moving. f has a CV of 0.7%, so it is
# a sensitive probe of p having changed. Read BOTH before believing any diversity gain.
from creativity_measure.flowmap_smc import posterior_z_ddpm_step

STRENGTH_LIST = [float(x) for x in os.environ.get("STRENGTH_LIST", "0,0.25,0.5,1.0").split(",")]
S_DATA_FLUX = 1.0150          # measured by the phase-1 health gate; NOT a guess (CLAUDE.md)
# Clones are born at the resample, and a3_1 m=2 was down to one ancestor by step 3. Probing at both a
# realistic-early and a mid split says whether any regeneration is possible at all, and by when it is
# already too late -- sigma_cond falls with sigma_{t'}, so the tail cannot repair a late collapse.
SPLIT_STEPS = [3, 8]
TS = [n / N_STEPS for n in range(N_STEPS + 1)]
STOCH_WINDOW = BASE_PROCESS["stoch_window"]


def _step_fn_for(t, inside):
    """`stoch_window` keys on where a step STARTS -- same rule the sampler uses."""
    lo, hi = STOCH_WINDOW
    return inside if (lo - 1e-9) <= t <= (hi + 1e-9) else flow_map_step


def _iterate(x, n_from, n_to, inside, gen):
    for n in range(n_from, n_to):
        x = _step_fn_for(TS[n], inside)(x, TS[n], TS[n + 1], flow_map=flow_map,
                                        schedule=SCHEDULE, generator=gen)
    return x


def _pair_stats(X):
    """Median / min pairwise L2. Independent FLUX latents sit at ~sqrt(2*d)*std ~ 362."""
    Xf = X.float().reshape(X.shape[0], -1)
    D = torch.cdist(Xf, Xf)
    iu = torch.triu_indices(D.shape[0], D.shape[0], 1)
    pd = D[iu[0], iu[1]]
    return float(pd.median()), float(pd.min())


strength_out = {}
if DO_STRENGTH:
    note(f"=== PHASE 6: strength sweep {STRENGTH_LIST} (lambda=0, M={M}, seed={SEED}) ===")
    note(f"    independent-draw reference distance ~ {math.sqrt(2 * C * H * W) * S_DATA_FLUX:.0f}")

    for strength in STRENGTH_LIST:
        inside = ddpm_step if strength == 0.0 else posterior_z_ddpm_step(
            s_data=S_DATA_FLUX, strength=strength)
        rec = {"strength": strength}

        # (A) BASE-PROCESS HEALTH. lambda = 0, so this is a plain draw from p under the new transition.
        # Distinct starts, no resampling (systematic resampling on uniform weights is the identity),
        # so this measures p itself: latent std against 1.0150 and E_p[f] against the a3_* runs.
        t0 = time.time()
        res = flowmap_smc_sample(
            reward, 0.0, M, **{**BASE_KW, "inside_step": inside}, mc_samples=1, eta=ETA,
            guid_window=(1.0, 1.0), ess_threshold=ESS_THRESHOLD, antithetic=True,
            seed=SEED, verbose=False,
        )
        rec["base_std"] = float(res.X.float().std())
        rec["base_mean"] = float(res.X.float().mean())
        rec["E_p_f"] = res.f_mean[-1]
        rec["base_pair_med"], rec["base_pair_min"] = _pair_stats(res.X)
        rec["base_X"] = res.X.cpu()
        note(f"  strength={strength:g}  BASE std={rec['base_std']:.4f} (want {S_DATA_FLUX}) "
             f"E_p[f]={rec['E_p_f']:.4f}  pairdist med={rec['base_pair_med']:.0f} "
             f"[{time.time() - t0:.0f}s]")

        # (B) CLONE-SEPARATION PROBE. The decisive one, and it needs no reward at all: run ONE particle
        # to the split, duplicate it M times -- exactly what a resample produces -- and let each clone
        # finish the trajectory under its own noise. If the transition regenerates semantic diversity,
        # these decode to different pictures; if not, `uniq/N` was never recoverable downstream.
        rec["clones"] = {}
        for split in SPLIT_STEPS:
            gen = torch.Generator(device=DEVICE).manual_seed(SEED + 500 + split)
            x1 = torch.randn((1, C * H * W), generator=gen, device=DEVICE, dtype=res.X.dtype)
            x1 = _iterate(x1, 0, split, inside, gen)
            clones = x1.expand(M, -1).contiguous()          # M identical particles, as after a resample
            xf = _iterate(clones, split, N_STEPS, inside, gen)
            med, mn = _pair_stats(xf)
            rec["clones"][split] = {"pair_med": med, "pair_min": mn,
                                    "std": float(xf.float().std()), "X": xf.cpu()}
            note(f"  strength={strength:g}  CLONES split@step{split} (t={TS[split]:.3f}) "
                 f"pairdist med={med:.0f} min={mn:.0f}  std={float(xf.float().std()):.4f}")
        strength_out[strength] = rec

    torch.save({"strength_out": strength_out, "config": {
        "M": M, "seed": SEED, "n_steps": N_STEPS, "ess_threshold": ESS_THRESHOLD,
        "stoch_window": STOCH_WINDOW, "s_data": S_DATA_FLUX, "split_steps": SPLIT_STEPS,
        "strength_list": STRENGTH_LIST, "base_process": BASE_PROCESS, "lam": 0.0,
    }}, os.path.join(RUN_DIR, f"strength_sweep_{RUN_TAG}.pt"))
    note(f"saved strength_sweep_{RUN_TAG}.pt")

    # --- images: one grid per probe, rows = strength ---------------------------------------------
    for what, key in (("base", None), *((f"clones_split{s}", s) for s in SPLIT_STEPS)):
        fig, axes = plt.subplots(len(STRENGTH_LIST), M,
                                 figsize=(2.0 * M, 2.0 * len(STRENGTH_LIST)), squeeze=False)
        for r, strength in enumerate(STRENGTH_LIST):
            X = (strength_out[strength]["base_X"] if key is None
                 else strength_out[strength]["clones"][key]["X"])
            imgs = decode(X.to(DEVICE))
            for c_ in range(M):
                axes[r][c_].imshow(imgs[c_].permute(1, 2, 0).numpy()); axes[r][c_].axis("off")
            axes[r][0].set_ylabel(f"s={strength:g}", fontsize=9)
            axes[r][0].axis("on"); axes[r][0].set_xticks([]); axes[r][0].set_yticks([])
        ttl = ("base process p (lambda=0)" if key is None
               else f"clones split at step {key} (t={TS[key]:.3f})")
        fig.suptitle(f"PHASE 6 -- {ttl} -- rows are posterior_z strength")
        fig.tight_layout()
        out = os.path.join(RUN_DIR, f"strength_{what}_{RUN_TAG}.png")
        fig.savefig(out, dpi=110, bbox_inches="tight"); plt.close(fig)
        note(f"wrote {os.path.basename(out)}")

    print(f"{'strength':>9} {'base std':>9} {'E_p[f]':>9} {'base pd':>8} "
          + " ".join(f"{'clone pd@' + str(s):>13}" for s in SPLIT_STEPS))
    for strength in STRENGTH_LIST:
        r = strength_out[strength]
        print(f"{strength:>9g} {r['base_std']:>9.4f} {r['E_p_f']:>9.4f} {r['base_pair_med']:>8.0f} "
              + " ".join(f"{r['clones'][s]['pair_med']:>13.0f}" for s in SPLIT_STEPS))

## Takeaways — a3_2 (latest)

Job 702190, L40S, 8.18 h. Branch 1 ($\hat v_{k} = \lambda R_{k}$), $M=8$, $K=4$, $N=16$, $\eta=1.5$,
`ess_threshold=0.5`, `guid_window=(0.0, 1.0)`, $\lambda = m \lambda_{s}$ with $\lambda_{s} = 140.09$, seed 101.
*(The cell below is the a3_1 record. Its "what to change next" list is resolved by this run: lowering
`ess_threshold` was tried and did not work.)*

### 1. ESS-triggered resampling is the wrong mechanism for this sampler

Dropping the threshold from 1.0 to 0.5 did exactly what it was meant to — $m=1$ fired **once** instead of
13 times — and the cloud degenerated anyway. Collapse is driven by the resampling *events themselves*,
not by how often they fire:

| $m$ | $\lambda$ | resamples fired | uniq/$M$ after each | final uniq/$M$ | min ESS/$M$ | $E_{q}[f]$ |
|---|---|---|---|---|---|---|
| 1 | 140.1 | **1** (step 5) | $1.00 \to 0.38$ | 0.38 (3 of 8) | 0.25 | 1.0047 |
| 2 | 280.2 | 8 (steps 1–6, 8, 10) | $0.38 \to 0.25 \to \mathbf{0.12}$ | 0.12 (1 of 8) | 0.13 | 0.9997 |
| 3 | 420.3 | 8 (steps 1–6, 8, 11) | $0.25$ held 6 resamples $\to \mathbf{0.12}$ | 0.12 (1 of 8) | 0.14 | 0.9990 |

**Number of resamples to reach a single surviving ancestor: 3 for $m=2$, 7 for $m=3$; $m=1$ never got
there but lost 5 of 8 ancestors on its single resample and stayed at 3 for the remaining 11 steps.**
uniq/$M$ is monotone non-increasing in every run — systematic resampling can only destroy lineages, and
nothing in the current step regenerates them, so any threshold merely sets the *rate* of an inevitable
collapse. Confirmed visually in `tilt_grid_a3_2.png`: $m=2$ and $m=3$ are eight copies of one image.

**Conclusion: the fix must be a different resampling/rejuvenation scheme, not a different threshold.**
The natural direction is something closer to Algorithm 1 (`adaptive_tempering_smc.py`), which pairs each
resample with a prior-preserving move that puts diversity back. Method still to be chosen.

### 2. The tilt is not yet strong enough — the samples are not creative

$E_{q}[f]$ = 1.0047 / 0.9997 / 0.9990 against $E_{p}[f] = 1.0062 \pm 0.0071$: at or *below* the untilted
mean at every $\lambda$. This is not purely a degeneracy artefact — the per-step trace shows $f$ climbing
under guidance ($m=2$ peaks at 1.045 around $t = 0.69$) and then decaying back to $\approx 1.00$ over the
last four steps. Whatever tilt the lookahead applies mid-trajectory is undone before $t = 1$.

### 3. The reward side is settled — do not re-open it

$R = 64$, `RandomRefs` uniform, and $\lambda_{s} = 140.09$ are **not** open parameters, and none of them
is what a3_1/a3_2 failed on:

- $R = 64$ sits on the rank-stability plateau (independent-draw $\tau = 0.960$, flat for $R \ge 32$).
  Moving to 32 saves only half the 19.1 min bank build — per-$\lambda$ cost is independent of $R$ — while
  breaking comparability with a3_1/a3_2. Moving to 256 costs 1.6 h and buys precision nothing reads.
- $\lambda_{s}$ carries $\pm 13\%$ from its 32 held-out samples. Irrelevant: the $m$ grid spans $3\times$
  and selection is on in-run `uniq/M`. Cached in `lam_s_R64_seed7.json`; never re-measure.
- $f(\text{refs}) = 63/64$ is an exact algebraic identity, not evidence about the model — see CLAUDE.md.

Full derivations in `notebooks/refset_auto_r/`. The open questions are all on the sampler side.

### 4. Recommended parameters for the next branch-1 run

> **RUN ONLY AFTER THE RESAMPLING METHOD IS REPLACED.** These settings on the current ESS-triggered
> resampler will reproduce a3_2 and waste ~10 h. The resampling change is the prerequisite, not a
> parallel experiment.

- $M = 16$, $K = 4$ — $M$ is the one lever never tested, and 8 particles in $d = 65536$ is nothing.
  Cost scales as $M K$: $\approx 5.0$ h per $\lambda$ on an L40S at 0.206 s/row, so budget **one $\lambda$
  per 9 h job**. If a cheap A/B against a3_2 is wanted first, keep $M = 8$ and change *only* the resampler.
- $\lambda = m \lambda_{s}$, $m \in \{2, 3\}$. Drop $m = 1$: a3_2 shows it is too weak to move $f$ even
  though it preserved the most diversity.
- Frozen for comparability: $N = 16$, $\eta = 1.5$, `guid_window=(0.0, 1.0)`, `seed=101`,
  `refs_R64_seed7.pt`, $\lambda_{s} = 140.09$. No reference rebuild, no recalibration.
- Read the gate before anything else: **uniq/$M \ge 0.5$ at $t = 1$** and **$E_{q}[f] \gtrsim 1.020$**
  ($+2\sigma$ over $E_{p}[f]$, the smallest detectable novelty gain). Below either, the run is another
  negative result and the images do not need looking at.
- Free 19 min/job: persist the `score_bank` alongside `refs_R64_seed7.pt` — currently only the latents are
  saved, so the $O(R)$ bank is rebuilt every job.

The branch-2 (Z-scored) counterpart is `notebooks/flowmap_smc_flux_normv/` (run nv_1), which degenerated
the same way — so this is a property of the resampler, not of the potential.


## Takeaways

**Headline: the machinery is correct and fast, but at $M=8$ with `ess_threshold=1.0` the sampler
collapses to a single trajectory at every $\lambda$ tested, and produces no measurable novelty gain.
This is a decisive negative result about the *operating point*, not about Algorithm 3.**

Job 700517, L40S, 6.95 h. $M=8$, $K=4$, $N=16$, $\eta=1.5$, `guid_window=(0.1, 0.9)`,
$\lambda = m\lambda_s$ with $\lambda_s = 140.09$, seed 101.

### What happened

| $m$ | $\lambda$ | min ESS/$M$ | final uniq/$M$ | $E_q[f]$ | latent diversity |
|---|---|---|---|---|---|
| 1 | 140.1 | 0.248 | **0.125** | 1.0125 | 58% of iid |
| 2 | 280.2 | 0.128 | **0.125** | 1.0065 | 33% of iid |
| 3 | 420.3 | 0.125 | **0.125** | 1.0001 | 20% of iid |
| — | *base $p$* | — | 1.0 | $1.0062 \pm 0.0071$ | 100% (by definition) |

`uniq/M = 0.125` is **one surviving ancestor out of eight**, at every $\lambda$. Per the CLAUDE.md
selection rule — largest $\lambda$ whose `uniq/N` and min ESS/$N$ stay above ~0.5 — **none of the three
qualifies**. The decoded grid shows this plainly: row 1 (untilted) is eight visibly different dogs;
rows 2–4 are eight copies of one dog each.

**There is no novelty gain.** $E_q[f]$ is non-monotone in $\lambda$ and every value sits within ~1$\sigma$
of $E_p[f] = 1.0062 \pm 0.0071$. With one effective particle these are single-draw estimates, so the
apparent *decrease* with $\lambda$ is noise, not signal — but the absence of an *increase* is real.

### Why — and it is not primarily $\lambda$

`resampled_history` is `R` at **all 13 guided steps** for all three $\lambda$, because
`ess_threshold=1.0` resamples unconditionally. Two effects compound:

1. **One lookahead update is already enough to crush ESS.** The *first* guided step lands at
   ESS/$M$ = 0.25 / 0.13 / 0.13 for $m$ = 1 / 2 / 3. In branch 1 ($\hat v_k = \lambda R_k$),
   `logsumexp` $\approx \lambda\max_k R_k$, so a reward spread of ~0.02 across particles becomes a
   weight spread of several nats. At $M=8$ that leaves 1–2 effective particles immediately. The
   per-step $V$ spread across particles stays large all run — 1.3–8.0 nats at $m=1$, up to 26 at $m=3$.
2. **Resampling every step converts that into permanent lineage loss.** Thirteen consecutive systematic
   resamplings on 8 particles cannot preserve ancestry no matter how mild the weights. Note $m=1$
   decays gradually (uniq 0.38 → 0.25 → 0.125 by step 9) while $m=3$ collapses at step 2 — the rate
   tracks $\lambda$, but the endpoint does not.

This is exactly what the plan predicted: *"the real levers against duplicate outputs are
`ess_threshold` — resample less often near the end — and $M$."*

> **Correction, and a trap to avoid.** An earlier draft of this cell argued that because ESS/$M$ sits at
> 0.5–1.0 from step 3 onward, a 0.5 threshold "would fire only 2–3 times". **That inference is invalid**
> and the number should not be trusted. Those ESS readings were produced *under* `ess_threshold=1.0`,
> which zeroes $U$ every step — so each one measures a single telescoped increment
> $V_{t+h} - V_t$, not the accumulated weight. With a 0.5 threshold $U$ accumulates across
> non-resampled steps and ESS necessarily falls faster, so the true firing count lies somewhere between
> 2 and 13 and cannot be read off this run. More generally: **an ESS history is only interpretable
> relative to the threshold that generated it.** (Relatedly, high ESS late in a3_1 partly reflects
> duplicated particles carrying identical weights — degeneracy inflating the very statistic meant to
> detect it, which is why `uniq/N` and not ESS is the primary signal.)

### Method notes worth keeping

- **`uniq/N` and geometric distinctness are different questions, and both are needed.** All three runs
  report uniq/$M$ = 0.125, but the *number of distinct latents* recovers to 7–8 of 8 by the last steps,
  because the stochastic tail perturbs the clones apart. Median pairwise latent distance (against the
  1.161 ratio measured on 64 iid draws from $p$) falls monotonically 58% → 33% → 20%. So the returned
  set is eight geometrically distinct points that are all *perturbations of one trajectory* — which is
  precisely why the images look near-identical while `n_distinct` looks healthy. Report ancestry for
  degeneracy, pairwise distance for what the images actually are, and never `n_distinct` alone.
- **The cost model is excellent.** 2.16 / 2.16 / 2.17 h per $\lambda$ against 2.17 h projected — 0.5%
  error, from the reference-bank measurement of 0.206 s/row.
- **Everything reproduced.** The Phase 1 gate returned `latent_std=1.0345`, `gap=0.1660` identically
  across three separate jobs and node allocations; the reference bank rebuilt in 19.1 min twice;
  $f(\text{refs}) = 63/64$ exactly. The determinism the SMC theory assumes is real here.
- **Did Algorithm 3 beat the pCN ladder?** *Unanswerable from this run.* One forward pass at
  $\lambda = 3\lambda_s$ costs 2.16 h against the ladder's ~14 h to reach $m_{\text{eff}} = 1.63$, so the
  **cost** claim holds emphatically. But a degenerate cloud is not a sample from $q_\lambda$, so there
  is nothing yet to compare on quality. That comparison needs a non-degenerate run.

### What to change next, in order

1. **`ess_threshold=0.5`** — cheapest direct test of lever (2), at near-identical cost. **This is run
   a3_2** (job 702178), together with the full `guid_window=(0.1, 1.0)`: a3_1 narrowed the window purely
   to buy wall clock, which is a confound while the method still fails, so it is restored until results
   are good. Reuses the same references and $\lambda_s$ — no recalibration, because `guid_window` was
   deliberately kept out of $p$.
2. **More particles.** $M=8$ is very small for SMC in $d = 65536$, and attacks lever (1) as well as (2);
   cost scales as $M\cdot K$, so $M=32, K=2$ or $M=16, K=4$ doubles the per-$\lambda$ cost to ~4.3 h.
3. **`use_full_normalized_v=True`** (the Z-scored branch) attacks lever (1) at its root: Z-scoring makes
   the twist scale-free, so a large $\lambda$ can no longer blow up the weight spread — that is what the
   separate `lam_normalized` knob exists for. Completely untested so far.
4. **Lower $\lambda$.** Not recommended first: $m=1$ already collapses, so $\lambda$ is not the
   binding constraint.

**Still open** (unchanged by this run): whether $\hat L$ / $\hat S$ move the weights at all at $K=4$ —
they were recorded per step and can be read off `l_mean` / `s_mean` in the saved results — and the
final `guid_window`, which can be re-swept against these same references at no extra calibration cost.